# Урок 7.2. Оркестратор на чистом Python

В уроке 7.1 мы выяснили, что командой роли работают лучше, чем в одном промпте, —
но только если у ролей проверяемые выходы. Сегодня строим всё по-настоящему:
команда «аналитик → разработчик → ревьюер» пишет рабочий код для «Пингвин.Хост»,
а решения «принято / дорабатывать / менять стратегию / сдаться» принимает
**оркестратор — обычный Python-код**.

🎯 **Цели урока:**

- Спроектировать обмен сообщениями: журнал, роли, телеметрия каждого вызова.
- Сделать судьёй не LLM, а **настоящие тесты** в изолированном процессе.
- Построить цикл доработки с лимитом — и увидеть вживую, как он буксует.
- Добавить эскалацию: смена стратегии вместо очередной итерации — и увидеть,
  как она побеждает.
- Обрабатывать ошибки агентов: мусор вместо кода, обрывы, бюджет команды.

**Что понадобится:** Ollama с qwen2.5:3b. Ограничения 3B — по-прежнему часть
сюжета, и сегодня они будут главным антагонистом.

## 1. Архитектура: четыре детали

```
             ┌────────────────────── ОРКЕСТРАТОР (обычный Python) ─────────────────────┐
             │                                                                         │
 задача ────▶│  [аналитик] ─спека─▶ [разработчик] ─код─▶ ⚙ ТЕСТЫ ─отчёт─▶ решение:     │
             │                          ▲                (subprocess)   принято?       │
             │                          │                               фикс-цикл?     │
             │                  [трассировщик]◀─── упавший кейс ────────эскалация?     │
             │                                                          человек?       │
             └─────────────────────────────────────────────────────────────────────────┘
                              всё пишется в журнал сообщений (общая память)
```

1. **Сообщение** — кто, кому, что и почём (телеметрия из урока 5.5 на каждый вызов).
2. **Журнал** — общая память команды: по нему можно отладить любой прогон
   (чёрный ящик агента из урока 3.7 — теперь для целой команды).
3. **Роли** — узкие системные промпты со своими бюджетами `num_predict` (урок 7.1).
4. **Оркестратор** — код, который двигает работу между ролями и принимает решения.

Заметьте, чего здесь **нет**: LLM-супервизора. Маршрут задачи известен заранее —
значит, его место в коде (то же правило, что в уроке 3.6: известный маршрут —
это граф/код, LLM решает только там, где маршрут зависит от входа).

In [ ]:
import json
import re
import subprocess
import sys
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path

import httpx

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen2.5:3b"


@dataclass
class Message:
    """Одна реплика в команде: кто, кому, что - и почём."""
    sender: str
    recipient: str
    kind: str          # spec / code / test_report / trace / decision
    content: str
    prompt_tokens: int = 0
    answer_tokens: int = 0
    seconds: float = 0.0


@dataclass
class TeamLog:
    """Журнал команды: общая память + телеметрия (рифма с логами урока 3.7)."""
    messages: list[Message] = field(default_factory=list)

    def add(self, msg: Message) -> None:
        self.messages.append(msg)
        cost = f" [{msg.answer_tokens} ток, {msg.seconds:.0f} c]" if msg.answer_tokens else ""
        print(f"  {msg.sender} -> {msg.recipient} ({msg.kind}){cost}")

    def llm_calls(self) -> int:
        return sum(1 for m in self.messages if m.answer_tokens)

    def total_tokens(self) -> tuple[int, int]:
        return (sum(m.prompt_tokens for m in self.messages),
                sum(m.answer_tokens for m in self.messages))

## 2. Роли: узкий промпт + свой бюджет

Агент здесь — минимальный: системный промпт, бюджет и журналирование. Никаких
фреймворков: весь «интеллект» команды — в промптах ролей и решениях оркестратора.

In [ ]:
class Agent:
    def __init__(self, name: str, system: str, num_predict: int, log: TeamLog):
        self.name, self.system, self.num_predict, self.log = name, system, num_predict, log

    def ask(self, user: str, recipient: str, kind: str) -> str:
        started = time.perf_counter()
        response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
            "model": MODEL,
            "messages": [{"role": "system", "content": self.system},
                         {"role": "user", "content": user}],
            "stream": False,
            "options": {"temperature": 0, "num_predict": self.num_predict},
        }, timeout=300)
        response.raise_for_status()
        data = response.json()
        content = data["message"]["content"]
        self.log.add(Message(
            sender=self.name, recipient=recipient, kind=kind, content=content,
            prompt_tokens=data.get("prompt_eval_count", 0),
            answer_tokens=data.get("eval_count", 0),
            seconds=time.perf_counter() - started,
        ))
        return content


def make_team(log: TeamLog) -> dict[str, Agent]:
    return {
        "analyst": Agent("аналитик",
                         "Ты — аналитик. Пиши кратко. НЕ пиши код.", 400, log),
        "developer": Agent("разработчик",
                           "Ты — Python-разработчик. Возвращай ТОЛЬКО код одним блоком ```python```.",
                           600, log),
        "tracer": Agent("трассировщик",
                        "Ты — код-ревьюер-трассировщик. Работай строго по чеклисту, не переписывай код.",
                        400, log),
    }

## 3. Судья, которому нельзя заговорить зубы

В уроке 7.1 лучший судья для тикета получился из чеклиста. Для **кода** судья
ещё лучше: интерпретатор Python с приёмочными тестами. Его не убедить красивым
объяснением — код либо проходит тесты, либо нет.

Два инженерных решения:

- **Запуск в отдельном процессе с таймаутом.** Код written by LLM — это
  недоверенный код (принцип урока 4.2). Для учебной локальной задачи достаточно
  `subprocess` + timeout (защита от вечного цикла); в проде — контейнер без сети,
  как CI-раннер.
- **Отчёт — интерфейс между кодом и моделью.** В отчёте должен быть виден
  **вход, факт и ожидание** (`got` / `expected`) — по нему модель будет чинить.
  Нечитаемый отчёт = слепой ревьюер (увидим в упражнении 1).

И привычка из урока 3.3 — **сначала калибруем судью**: подсунем заведомо
сломанный код и проверим, что харнес его ловит, а не молчит.

In [ ]:
def extract_code(text: str) -> str | None:
    """Валидация формата: из ответа агента достаём блок кода (или None)."""
    match = re.search(r"```(?:python)?\s*(.+?)```", text, re.DOTALL)
    return match.group(1).strip() if match else None


HARNESS = """
import json
results = []
{code}
for expr, expected in {tests!r}:
    try:
        got = eval(expr)
        results.append({{"case": expr, "ok": got == expected, "got": repr(got), "expected": expected}})
    except Exception as exc:
        results.append({{"case": expr, "ok": False, "got": f"{{type(exc).__name__}}: {{exc}}", "expected": expected}})
for expr in {errors!r}:
    try:
        eval(expr)
        results.append({{"case": expr + " => ValueError", "ok": False, "got": "исключения не было", "expected": "ValueError"}})
    except ValueError:
        results.append({{"case": expr + " => ValueError", "ok": True, "got": "ValueError", "expected": "ValueError"}})
    except Exception as exc:
        results.append({{"case": expr + " => ValueError", "ok": False, "got": f"{{type(exc).__name__}}", "expected": "ValueError"}})
print(json.dumps(results, ensure_ascii=False))
"""


def run_tests(code_str: str, tests: list, error_cases: list) -> list[dict]:
    """Гоняем код LLM в ОТДЕЛЬНОМ процессе с таймаутом (недоверенный код!)."""
    source = HARNESS.format(code=code_str, tests=tests, errors=error_cases)
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False,
                                     encoding="utf-8") as f:
        f.write(source)
        path = f.name
    try:
        proc = subprocess.run([sys.executable, "-X", "utf8", path],
                              capture_output=True, text=True, encoding="utf-8",
                              timeout=10)
        if proc.returncode != 0:   # код не загрузился (SyntaxError и т.п.)
            return [{"case": "<загрузка кода>", "ok": False,
                     "got": proc.stderr.strip()[-200:], "expected": "импорт без ошибок"}]
        return json.loads(proc.stdout)
    except subprocess.TimeoutExpired:
        return [{"case": "<выполнение>", "ok": False,
                 "got": "timeout 10 c (вечный цикл?)", "expected": "завершение"}]
    finally:
        Path(path).unlink(missing_ok=True)


def report_text(results: list[dict]) -> str:
    return "\n".join(
        f"[{'PASS' if r['ok'] else 'FAIL'}] {r['case']} -> {r['got']}"
        f" (ожидалось {r['expected']})" for r in results)


# калибровка судьи: сломанный код обязан провалиться
broken = "def normalize_phone(text):\n    return text"
demo = run_tests(broken, [('normalize_phone("8 912")', "7912")], [])
print(report_text(demo))
assert not demo[0]["ok"], "судья пропустил заведомо сломанный код!"
print("Судья откалиброван: брак ловит.")

## 4. Задача и приёмочные тесты

Задача из бэклога «Пингвин.Хост»: биллинг рассылает SMS, а телефоны в базе
клиенты вводили как попало. Нужна нормализация.

Приёмочные тесты пишет **человек** (мы), а не команда агентов: критерий приёмки
не должен сочинять тот, кого по нему принимают, — иначе команда с удовольствием
примет саму себя (конфликт интересов, знакомый по «мягким чекерам» урока 5.2).

In [ ]:
TASK = """Функция Python normalize_phone(text: str) -> str: приводит российский
номер телефона к каноничному виду - строка из 11 цифр БЕЗ разделителей,
начинается с 7. Вход бывает в разнобой: "+7 (912) 345-67-89", "8-912-345-67-89",
с пробелами. Номер, начинающийся с 8, - это тот же номер с кодом 7.
Если после очистки не получается корректный номер (11 цифр, первая 7 или 8) -
подними ValueError."""

TESTS = [
    ('normalize_phone("+7 (912) 345-67-89")', "79123456789"),
    ('normalize_phone("8-912-345-67-89")', "79123456789"),
    ('normalize_phone("79123456789")', "79123456789"),
    ('normalize_phone("8 (495) 123 45 67")', "74951234567"),
]
ERROR_CASES = ['normalize_phone("12345")', 'normalize_phone("нет телефона")']
print(f"приёмка: {len(TESTS)} кейсов значений + {len(ERROR_CASES)} кейса ValueError")

## 5. Оркестратор: конвейер + фикс-цикл + эскалация

Политика оркестратора — конечный автомат с тремя выходами:

```
 спека -> код v1 -> ТЕСТЫ ─ зелёные ──────────────────────────────▶ ПРИНЯТО
                      │ красные
                      ▼
              фикс-цикл (<= 2 раундов): трассировка -> правка -> ТЕСТЫ
                      │ всё ещё красные
                      ▼
              ЭСКАЛАЦИЯ: переписать С ЧИСТОГО ЛИСТА по диагнозу,
              СТАРЫЙ КОД НЕ ПОКАЗЫВАЕМ -> ТЕСТЫ
                      │ красные
                      ▼
              новый фикс-цикл для нового кода (<= 2 раундов)
                      │ всё ещё красные
                      ▼
              СДАТЬСЯ ЧЕСТНО: вернуть человеку артефакты и журнал
```

Обратите внимание на предпоследний блок: эскалация — это **новая попытка
конвейера, а не одиночный выстрел**. Переписанный с нуля код — обычно другой
и ломается в других местах; лишать его права на доработку так же глупо, как
доработывать старый до бесконечности. (Мы сначала построили оркестратор без
этого блока — и он сдался, держа в руках код, которому не хватало одной строчки.)

Два неочевидных решения, оба выстраданы прогонами:

- **Трассировщик вместо «найди проблемы».** Свободное ревью 3B-модели генерирует
  правдоподобные общие слова (урок 7.1). Чеклист «выпиши значение каждой
  переменной для конкретного упавшего входа» — тот же приём, что чинил судью
  тикетов, — заставляет модель смотреть в код, а не в облака.
- **Эскалация прячет старый код.** После пары неудачных правок модель
  анкорится: таскает сломанный каркас из версии в версию, меняя мелочи. Чистый
  лист с диагнозом (но без кода!) обрывает якорь. Смена стратегии — тоже решение
  оркестратора, и принимает его код, а не модель.

In [ ]:
MAX_FIX_ROUNDS = 2

def run_team(task: str, tests: list, error_cases: list) -> dict:
    """Полный прогон команды. Возвращает артефакты + вердикт."""
    log = TeamLog()
    team = make_team(log)
    started = time.perf_counter()

    def test_and_report(code_str: str) -> tuple[list[dict], str, int]:
        results = run_tests(code_str, tests, error_cases)
        passed = sum(r["ok"] for r in results)
        log.add(Message("тесты", "оркестратор", "test_report",
                        report_text(results)))
        print(f"    тесты: {passed}/{len(results)}")
        return results, report_text(results), passed

    def finish(verdict: str, code_str: str, report: str) -> dict:
        p_tok, a_tok = log.total_tokens()
        return {"verdict": verdict, "code": code_str, "report": report,
                "llm_calls": log.llm_calls(), "prompt_tokens": p_tok,
                "answer_tokens": a_tok,
                "seconds": round(time.perf_counter() - started, 1),
                "log": log}

    def fix_cycle(code_str: str, results, report):
        """До MAX_FIX_ROUNDS раундов «трассировка -> правка -> тесты»."""
        passed = sum(r["ok"] for r in results)
        trace = ""
        for _ in range(MAX_FIX_ROUNDS):
            if passed == len(results):
                break
            first_fail = next(r for r in results if not r["ok"])
            trace = team["tracer"].ask(
                f"Код:\n```python\n{code_str}\n```\n"
                f"Полный отчёт тестов:\n{report}\n\n"
                f"Разбери подробно первый упавший кейс: {first_fail['case']} "
                f"дал {first_fail['got']}, ожидалось {first_fail['expected']}.\n\n"
                "Заполни чеклист, подставляя этот вход ШАГ ЗА ШАГОМ:\n"
                "1. Выпиши значение каждой промежуточной переменной для этого входа.\n"
                "2. На какой строке значение впервые расходится с нужным?\n"
                "3. Причина одним предложением и исправление одним предложением.\n"
                "4. Посмотри на ОСТАЛЬНЫЕ FAIL в отчёте: какие ещё требования "
                "не выполнены? Перечисли каждое одной строкой.",
                "разработчик", "trace")
            fixed = team["developer"].ask(
                f"Задача:\n{task}\n\nТвой код не прошёл тесты.\n"
                f"```python\n{code_str}\n```\n\nОтчёт тестов:\n{report}\n\n"
                f"Трассировка ревьюера:\n{trace}\n\n"
                "Верни исправленную функцию целиком. Только код.", "тесты", "code")
            code_str = extract_code(fixed) or code_str
            results, report, passed = test_and_report(code_str)
        return code_str, results, report, passed, trace

    # --- конвейер -------------------------------------------------------------
    spec = team["analyst"].ask(
        f"Задача:\n{task}\n\nВыпиши 4-6 требований и 3-4 крайних случая списками.",
        "разработчик", "spec")

    dev_answer = team["developer"].ask(
        f"Задача:\n{task}\n\nСпецификация аналитика:\n{spec}\n\n"
        "Напиши функцию. Только код.", "тесты", "code")
    code_str = extract_code(dev_answer)
    if code_str is None:                       # агент вернул прозу вместо кода
        return finish("REJECTED: разработчик не вернул код", "", "")

    results, report, passed = test_and_report(code_str)

    # --- фикс-цикл v1 (лимит! - рифма max_steps из урока 3.5) ------------------
    code_str, results, report, passed, trace = fix_cycle(code_str, results, report)
    if passed == len(results):
        return finish("ACCEPTED", code_str, report)

    # --- эскалация: чистый лист, диагноз есть, старого кода НЕТ ----------------
    print("    оркестратор: фикс-цикл исчерпан -> эскалация (чистый лист)")
    rebuilt = team["developer"].ask(
        f"Задача:\n{task}\n\nПредыдущая попытка команды провалила тесты:\n"
        f"{report}\n\nДиагноз ревьюера:\n{trace}\n\n"
        "Напиши функцию ЗАНОВО, с чистого листа, учтя диагноз. Только код.",
        "тесты", "code")
    code_str = extract_code(rebuilt) or code_str
    results, report, passed = test_and_report(code_str)

    # --- новый код заслуживает свой фикс-цикл -----------------------------------
    code_str, results, report, passed, _ = fix_cycle(code_str, results, report)
    if passed == len(results):
        return finish("ACCEPTED (после эскалации)", code_str, report)

    return finish("REJECTED: нужен человек", code_str, report)

In [ ]:
try:
    outcome = run_team(TASK, TESTS, ERROR_CASES)
    print(f"\nВЕРДИКТ: {outcome['verdict']}")
    print(f"Цена: {outcome['llm_calls']} вызовов LLM, "
          f"{outcome['prompt_tokens']}+{outcome['answer_tokens']} ток., "
          f"{outcome['seconds']} c")
    print("\nФинальный код:\n" + outcome["code"])
except httpx.HTTPError as exc:
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}]")

Разбор моего прогона (детали у вас будут другими, картина — типичной):

- **v1: 4/6 с первой попытки.** Разработчик почти справился: чистая структура,
  дальше дело за двумя кейсами со скобками.
- **Фикс-цикл: 4/6 → 4/6.** Трассировщик каждый раз верно называл строку с
  очисткой, а чеклист-пункт «какие ещё требования не выполнены» даже внёс в код
  правило 8→7. Но главный якорь — символьный класс `[^0-9()-+]`, который
  *оставляет* скобки, плюсы и дефисы вместо того, чтобы убрать, — разработчик
  протаскивал через все версии.
- **Эскалация: снова 4/6.** Вот это самое поучительное: даже с чистого листа
  модель при `temperature: 0` воспроизвела **тот же регэксп** — это её «модальный
  ответ» на такую формулировку, и никакие наши подсказки его не перебили.
  Чистый лист лечит анкоринг на *контексте*, но не анкоринг в *весах*.
- **REJECTED за 11 вызовов, ~12 тыс. токенов и ~70 секунд.** Оркестратор
  исчерпал политику и честно передал смену — не выдав сломанный код за готовый.

Заметьте, чего НЕ случилось: бесконечного цикла, тихой деградации, «приняли,
потому что ревьюер похвалил». Все решения принимал код по показаниям тестов.
(В других наших прогонах команда с этой же политикой доходила до 6/6 сама —
у 3B-моделей грань между «добила» и «сдалась» проходит от прогона к прогону;
именно поэтому ветка отказа обязана быть полноценной.)

## 6. Ветка REJECTED в действии: человек с журналом

«Нужен человек» — не тупик, а передача смены. Человек открывает журнал и видит:
спека готова, тесты написаны, код почти рабочий, диагноз поставлен — осталась
одна строчка, на которой модель зациклилась. Побудем этим человеком (у вас
финальный код команды может отличаться — читайте СВОЙ журнал):

In [ ]:
# Человек читает журнал: диагноз «после очистки остаются нецифровые символы»
# повторялся в каждой трассировке. Ремонт - одна идея: оставить ТОЛЬКО цифры.
human_code = '''
def normalize_phone(text: str) -> str:
    digits = "".join(ch for ch in text if ch.isdigit())   # только цифры, без исключений
    if len(digits) != 11 or digits[0] not in "78":
        raise ValueError(f"не похоже на номер: {text!r}")
    return "7" + digits[1:]                               # 8XXXXXXXXXX -> 7XXXXXXXXXX
'''

final = run_tests(human_code, TESTS, ERROR_CASES)
print(report_text(final))
print(f"\nприёмка: {sum(r['ok'] for r in final)}/{len(final)} - "
      f"{'ПРИНЯТО' if all(r['ok'] for r in final) else 'ещё нет'}")

6/6 — та же приёмка, что гоняла команду, приняла человеческую версию. Минута
работы — потому что журнал команды превратил задачу «напиши функцию» в задачу
«прочитай диагноз и поправь одну идею». В этом и есть честная ценность
мульти-агентной системы на маленькой модели: она не заменяет инженера — она
**готовит ему рабочее место**: спека, тесты, версии, диагнозы. А когда модель
подрастёт (или вы подключите модель побольше уроком 7.4), та же самая
оркестрация начнёт добегать до ACCEPTED сама — архитектура не меняется.

## 7. Что здесь важно запомнить

**Отчёт тестов — это интерфейс.** Цикл сходится, только если причина падения
*читается* из отчёта: `got '89...' / expected '79...'` чинится, а голый
`KeyError` из глубины регэкспа — нет (проверите в упражнении 1). Проектируя
харнес, вы проектируете зрение своего ревьюера.

**Анкоринг — реальная болезнь.** Модель, которой показывают её сломанный код,
правит его косметически, версия за версией. Дешёвое лекарство — не «ещё одна
итерация», а смена стратегии: чистый лист с диагнозом. Заметьте, что решение
об эскалации принял код по счётчику раундов — модель сама из колеи не выходит.

**Лимиты — везде.** Лимит фикс-раундов, таймаут на выполнение кода, бюджет
`num_predict` у каждой роли. Команда без лимитов на маленькой модели — это
генератор бесконечных вежливых доработок (урок 3.5 предупреждал про max_steps —
здесь всё то же, только этажом выше).

**Честный отказ — тоже результат.** Мы только что видели это вживую: ветка
«REJECTED: нужен человек» с журналом превратила ремонт в минутное дело.
Мульти-агентная система, которая умеет сдаваться, полезнее той, которая
имитирует успех.

## 8. Обработка ошибок агентов: чеклист оркестратора

Наш `run_team` уже обрабатывает главные, вот полный список поверхностей отказа:

| Отказ | Симптом | Лечение в оркестраторе |
|---|---|---|
| проза вместо кода | `extract_code -> None` | retry с ужесточением («ТОЛЬКО код»), потом REJECTED |
| кривой код | тесты: `<загрузка кода>` | обычный фикс-цикл: отчёт покажет SyntaxError |
| вечный цикл в коде | timeout харнеса | отчёт «timeout» -> фикс-цикл |
| обрыв ответа | `done_reason=length` | поднять num_predict роли или сократить вход |
| зависший Ollama | `httpx.TimeoutException` | retry с backoff — готовый клиент из урока 5.3 |
| перерасход бюджета | счётчики журнала | стоп-кран: max вызовов/токенов на прогон (упр. 3) |

Общий принцип — тот же, что в уроке 3.5: ошибка инструмента возвращается в цикл
**текстом**, а решает всегда код. LLM никогда не является последней линией обороны.

## ⚠️ Частые ошибки

1. **LLM-судья там, где есть код-судья.** Если результат проверяется тестами,
   компилятором или схемой — проверяйте ими. LLM-ревьюер остаётся для того,
   что кодом не проверить (читаемость, стиль).
2. **Нечитаемый отчёт тестов.** «AssertionError» без входа и ожидания — ревьюер
   слеп, цикл буксует. Вход/got/expected — минимум.
3. **Показывать старый код при эскалации.** Анкоринг тянет сломанный каркас
   в «новую» версию. Чистый лист — значит чистый.
4. **Доверять приёмочные тесты команде.** Кто пишет критерий — тот не проходит
   по нему приёмку. Тесты-приёмка — от человека (или хотя бы от другой системы).
5. **exec кода агента в своём процессе.** Вечный цикл повесит ноутбук, а
   `import os` сделает что похуже. Отдельный процесс + таймаут — минимум.
6. **Журналировать только финал.** Когда команда выдала мусор, отлаживать нужно
   по промежуточным сообщениям. Журнал — с первого дня, а не после инцидента.

## Упражнения

1. **Непрозрачный отчёт.** Прогоните `run_team` на задаче parse_size из урока 7.1
   (тесты: `("512 МБ", 536870912)`, `("1.5 ГБ", 1610612736)`, `("2 ТБ", 2199023255552)`,
   `("100КБ", 102400)`, `("0.5 КБ", 512)`, `("777", 777)`). Сойдётся ли команда?
   Посмотрите на отчёты: чем `KeyError: '.5'` из глубины регэкспа отличается —
   с точки зрения ревьюера — от `got '89...' expected '79...'`?

2. **Роль тест-инженера.** Добавьте в команду роль, которая по спеке аналитика
   генерирует ДОПОЛНИТЕЛЬНЫЕ тесты (наши приёмочные остаются от человека!).
   Валидируйте её выход кодом: каждый сгенерированный кейс должен быть валидным
   выражением (`ast.parse`) — и не должен противоречить приёмочным.

3. **Стоп-кран бюджета.** Добавьте в `run_team` параметр `max_llm_calls` (и/или
   `max_tokens`): при превышении — немедленный `REJECTED: бюджет исчерпан`
   с журналом. Рифма с `run_agent_budget` из урока 3.5.

In [ ]:
# Решение 1: та же команда, непрозрачная задача (это долгий прогон, ~1-2 минуты)
PARSE_TASK = """Функция Python parse_size(text: str) -> int: переводит человекочитаемый
размер диска в число байтов (множитель 1024). Единицы: КБ, МБ, ГБ, ТБ (русскими
буквами). Пробел между числом и единицей необязателен. Число может быть дробным.
Голое число без единицы - это уже байты."""

PARSE_TESTS = [
    ('parse_size("512 МБ")', 536870912),
    ('parse_size("1.5 ГБ")', 1610612736),
    ('parse_size("2 ТБ")', 2199023255552),
    ('parse_size("100КБ")', 102400),
    ('parse_size("0.5 КБ")', 512),
    ('parse_size("777")', 777),
]

try:
    outcome2 = run_team(PARSE_TASK, PARSE_TESTS, [])
    print(f"\nВЕРДИКТ: {outcome2['verdict']}")
    print(f"Цена: {outcome2['llm_calls']} вызовов, {outcome2['seconds']} c")
    print("\nПоследний отчёт:\n" + outcome2["report"])
except httpx.HTTPError as exc:
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}]")

**Разбор решения 1** (мой прогон): 0/6 на всех раундах, включая эскалацию, —
и посмотрите на отчёт: `ValueError: too many values to unpack (expected 2)`.
Это падение **внутри** кода, и оно не говорит ни «какая строка получилась»,
ни «что с ней не так» — только что где-то распаковка не сошлась. Сравните с
`got '89123456789' / expected '79123456789'` из основной задачи: там диф читается
глазами, здесь — нечего читать. Ревьюер-LLM слеп ровно настолько, насколько слеп
отчёт; parse_size с 3B у нас не сходился ни разу. Мораль упражнения: прежде чем
усиливать агентов, усильте наблюдаемость судьи (например, кейсы с ожидаемым
типом ошибки и промежуточным значением) — это дешевле и надёжнее.

**Решение 2 (эскиз).** Роль: «Ты — тест-инженер. По спецификации выпиши 5 новых
тестов СТРОГО в формате `("выражение", ожидание)`...». Дальше валидация кодом:
`ast.literal_eval` на каждую пару, `ast.parse` на выражение, и прогон против
финального кода **отдельно** от приёмочных: сгенерированные тесты — это сигнал
(«кандидаты в баги»), а не приёмка. Расхождение кандидата с приёмочным тестом
решается в пользу приёмочного.

**Решение 3 (эскиз).**

```python
def run_team(..., max_llm_calls: int = 12):
    ...
    def guard() -> bool:
        return log.llm_calls() < max_llm_calls
    # перед КАЖДЫМ team[...].ask(...):
    if not guard():
        return finish("REJECTED: бюджет исчерпан", code_str, report)
```

Смысл не в конкретном числе, а в том, что предохранитель срабатывает **до**
вызова, и команда не может его «уговорить» — бюджет живёт в коде.

## Мини-квиз

**1. Почему судьёй в этом уроке работают тесты, а не LLM-ревьюер, как в 7.1?**

<details><summary>Ответ</summary>

Для кода существует объективный исполняемый критерий — им и надо судить: тесты
нельзя убедить красивым объяснением, и они бесплатны (0 токенов). LLM-судья —
для решений без исполняемого критерия (релевантность, тон). Правило: код судит
всё, что может; LLM — остальное.
</details>

**2. Фикс-цикл три раунда подряд меняет по строчке, тесты красные. Что делает зрелый оркестратор?**

<details><summary>Ответ</summary>

Меняет стратегию, а не продолжает: это анкоринг. Эскалация — переписать с чистого
листа по диагнозу, не показывая старый код; если и это не помогло — честный
REJECTED с журналом для человека. Решение принимает счётчик в коде.
</details>

**3. Зачем прятать старый код при эскалации — он же содержит полезные наработки?**

<details><summary>Ответ</summary>

В том и проблема: модель не отличает наработки от бага и утаскивает каркас
целиком (якорь). Диагноз ревьюера переносит полезное знание («пробелы и дефисы
не удалены», «8 надо менять на 7») без сломанной реализации.
</details>

**4. Чем плох харнес, который на падении печатает только «AssertionError, кейс 3»?**

<details><summary>Ответ</summary>

Ревьюер-LLM чинит по отчёту; без входа, полученного и ожидаемого значений он
может только гадать. Отчёт тестов — интерфейс между судьёй и командой: вход /
got / expected — обязательные поля.
</details>

**5. Почему приёмочные тесты не поручить аналитику команды — он же лучше всех знает требования?**

<details><summary>Ответ</summary>

Конфликт интересов: критерий приёмки, написанный проверяемой стороной, дрейфует
под её же решение (и ошибки аналитика попадут и в код, и в тесты — самопринятие
гарантировано). Приёмка приходит извне: от человека или независимой системы.
</details>

**6. Команда вернула «REJECTED: нужен человек». Это провал архитектуры?**

<details><summary>Ответ</summary>

Нет, это её рабочая ветка: система знает границы своей компетенции и отдаёт
человеку журнал со спекой, версиями кода и отчётами — полуфабрикат, экономящий
время. Провал — это молчаливая выдача неработающего кода как готового.
</details>

## 🎓 Итоги

- Оркестратор — это **код**: маршрут, лимиты, эскалация и вердикт живут в Python,
  LLM-роли решают только свои узкие задачи.
- Журнал сообщений с телеметрией = общая память + отладка команды (чёрный ящик 3.7).
- Судья кода — **тесты в отдельном процессе с таймаутом**; судью калибруем до
  использования; отчёт тестов проектируем как интерфейс (вход/got/expected).
- Фикс-цикл с 3B буксует (анкоринг); эскалация с чистым листом помогает, но не
  гарантирует: в нашем прогоне модальный ответ модели пережил даже её, и команда
  честно сдалась.
- Все лимиты — в коде: раунды, таймауты, бюджеты. Честный REJECTED с журналом —
  полноценный результат: человек починил одну строчку за минуту, и та же приёмка
  дала 6/6.

**Дальше — урок 7.3:** та же команда на фреймворке CrewAI: Agent/Task/Crew,
подключение к Ollama — и честное сравнение с нашими 150 строками.

## 📚 Что почитать

- [Anthropic: Building effective agents](https://www.anthropic.com/research/building-effective-agents) — «простые компонуемые паттерны» вместо фреймворков
- [SWE-bench](https://arxiv.org/abs/2310.06770) — как всерьёз меряют «команды, пишущие код»
- [Reflexion](https://arxiv.org/abs/2303.11366) — самокоррекция агентов по сигналам среды (наша трассировка — её родня)
